# Rib Waveguide Interface: Generalized-Metric Derivation

Goal: derive and test a clean interface S-matrix workflow in one consistent overlap convention.

## 1) Conventions

We use one modal overlap operator `ip(m1, m2)` everywhere (mode solving, overlap matrices, interface solve, diagnostics).

For left basis `L_i` and right basis `R_j` define:
- `G_LL[i,j] = ip(L_i, L_j)`
- `G_RR[i,j] = ip(R_i, R_j)`
- `O_LR[i,j] = ip(L_i, R_j)`
- `O_RL[i,j] = ip(R_i, L_j)`

Generalized page-73 interface blocks:
- `T_LR = 2 * pinv(O_LR + O_RL.T) @ G_LL`
- `R_LL = 0.5 * pinv(G_LL) @ (O_RL.T - O_LR) @ T_LR`
- `T_RL = 2 * pinv(O_RL + O_LR.T) @ G_RR`
- `R_RR = 0.5 * pinv(G_RR) @ (O_LR.T - O_RL) @ T_RL`

Assemble the standard 2-port S-matrix:
`[a_L^- ; a_R^+] = [[R_LL, T_RL], [T_LR, R_RR]] [a_L^+ ; a_R^-]`

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

import meow as mw

In [ ]:
# Geometry (same rib-waveguide setup as rib_waveguide.ipynb)
wl = 1.0
widths = [0.8, 1.0]
t_slab = 0.2
t_soi = 1.5
t_ox = 0.2
n_Si = 3.4
n_SiO2 = 1.4
w_sim = 3.0
h_sim = 2.5
y_bot = -0.5

si = mw.IndexMaterial(n=n_Si, name="Si", meta={"color": "orange"})
sio2 = mw.IndexMaterial(n=n_SiO2, name="SiO2", meta={"color": "steelblue"})

env = mw.Environment(wl=wl, T=25.0)
mesh = mw.Mesh2D(
    x=np.linspace(-w_sim / 2, w_sim / 2, 101),
    y=np.linspace(y_bot, y_bot + h_sim, 101),
    # Keep no PML for close Lumerical comparison.
    # num_pml=(10, 10),
)


def make_cross_section(w):
    structures = [
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=y_bot,
                y_max=t_slab + t_ox,
            ),
            mesh_order=2,
        ),
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w / 2 - t_ox / 2,
                x_max=w / 2 + t_ox / 2,
                y_min=t_slab + t_ox,
                y_max=t_soi + t_ox,
            ),
            mesh_order=2,
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=0.0,
                y_max=t_slab,
            ),
            mesh_order=1,
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w / 2,
                x_max=w / 2,
                y_min=t_slab,
                y_max=t_soi,
            ),
            mesh_order=1,
        ),
    ]
    return mw.CrossSection(structures=structures, mesh=mesh, env=env)


cs_left = make_cross_section(widths[0])
cs_right = make_cross_section(widths[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs_left._visualize(ax=axes[0], show=False)
axes[0].set_title(f"Left (w={widths[0]} um)")
cs_right._visualize(ax=axes[1], show=False)
axes[1].set_title(f"Right (w={widths[1]} um)")
plt.tight_layout()
plt.show()

In [ ]:
# Choose ONE overlap convention and keep it everywhere.
ip_kwargs = {
    "symmetric": True,
    "conjugate": False,
    "ignore_pml": True,
    "interpolation": None,
}
ip = partial(mw.inner_product, **ip_kwargs)

num_modes = 100
modes_left = mw.compute_modes(cs_left, num_modes=num_modes, inner_product=ip)
modes_right = mw.compute_modes(cs_right, num_modes=num_modes, inner_product=ip)

print("ip kwargs:", ip_kwargs)
print(f"computed modes: left={len(modes_left)}, right={len(modes_right)}")

In [ ]:
def overlap_cross(modes_a, modes_b, op):
    M = np.zeros((len(modes_a), len(modes_b)), dtype=np.complex128)
    for i, ma in enumerate(modes_a):
        for j, mb in enumerate(modes_b):
            M[i, j] = op(ma, mb)
    return M


def hermitianize(M):
    return 0.5 * (M + M.conj().T)


def regularize_metric(G, rel_floor=1e-10, abs_floor=1e-12):
    Gh = hermitianize(G)
    w, V = np.linalg.eigh(Gh)
    floor = max(abs_floor, rel_floor * max(1.0, float(np.max(np.abs(w)))))
    w_clip = np.maximum(w.real, floor)
    G_reg = (V * w_clip) @ V.conj().T
    G_inv = V @ np.diag(1.0 / w_clip) @ V.conj().T
    return G_reg, G_inv, w, w_clip, floor


def metric_passivity_residual(S, G_in, G_out):
    M = hermitianize(S.conj().T @ G_out @ S - G_in)
    return float(np.max(np.linalg.eigvalsh(M).real))

In [ ]:
# Build overlap/metric matrices.
G_LL = overlap_cross(modes_left, modes_left, ip)
G_RR = overlap_cross(modes_right, modes_right, ip)
O_LR = overlap_cross(modes_left, modes_right, ip)
O_RL = overlap_cross(modes_right, modes_left, ip)

G_LL_reg, G_LL_inv, wL, wL_clip, floorL = regularize_metric(G_LL)
G_RR_reg, G_RR_inv, wR, wR_clip, floorR = regularize_metric(G_RR)

# Generalized page-73 blocks.
T_LR = 2.0 * np.linalg.pinv(O_LR + O_RL.T) @ G_LL_reg
R_LL = 0.5 * G_LL_inv @ (O_RL.T - O_LR) @ T_LR

T_RL = 2.0 * np.linalg.pinv(O_RL + O_LR.T) @ G_RR_reg
R_RR = 0.5 * G_RR_inv @ (O_LR.T - O_RL) @ T_RL

S_if = np.block([[R_LL, T_RL], [T_LR, R_RR]])

print("shapes:")
print("  G_LL, G_RR:", G_LL.shape, G_RR.shape)
print("  O_LR, O_RL:", O_LR.shape, O_RL.shape)
print("  S_if:", S_if.shape)
print(f"cond(O_LR + O_RL^T) = {np.linalg.cond(O_LR + O_RL.T):.3e}")
print(f"cond(O_RL + O_LR^T) = {np.linalg.cond(O_RL + O_LR.T):.3e}")

## 3) Equation Delta vs `rib_waveguide.ipynb` Form

Compare the generalized-metric derivation above against the simpler page-73-style implementation used in the exploratory notebook.

In [ ]:
# Compare with simpler page-73-style equations used in rib_waveguide.ipynb
# (without explicit G factors in T and R definitions).

T_LR_simple = 2.0 * np.linalg.pinv(O_LR + O_RL.T)
R_LL_simple = 0.5 * (O_RL.T - O_LR) @ T_LR_simple

T_RL_simple = 2.0 * np.linalg.pinv(O_RL + O_LR.T)
R_RR_simple = 0.5 * (O_LR.T - O_RL) @ T_RL_simple

S_simple = np.block([[R_LL_simple, T_RL_simple], [T_LR_simple, R_RR_simple]])

N = len(modes_left)
G_in_local = np.block(
    [
        [G_LL_reg, np.zeros((N, N), complex)],
        [np.zeros((N, N), complex), G_RR_reg],
    ]
)
G_out_local = G_in_local.copy()
res_passive_gen = metric_passivity_residual(S_if, G_in_local, G_out_local)
res_passive_simple = metric_passivity_residual(S_simple, G_in_local, G_out_local)

print("delta to generalized-metric formulation")
print(f"||T_LR - T_LR_simple||_F = {np.linalg.norm(T_LR - T_LR_simple):.3e}")
print(f"||R_LL - R_LL_simple||_F = {np.linalg.norm(R_LL - R_LL_simple):.3e}")
print(f"||T_RL - T_RL_simple||_F = {np.linalg.norm(T_RL - T_RL_simple):.3e}")
print(f"||R_RR - R_RR_simple||_F = {np.linalg.norm(R_RR - R_RR_simple):.3e}")
print(f"||S_if - S_simple||_F     = {np.linalg.norm(S_if - S_simple):.3e}")
print(f"metric passivity residual (generalized) = {res_passive_gen:.3e}")
print(f"metric passivity residual (simple)      = {res_passive_simple:.3e}")

## 2) Sanity Checks

For left incidence (`a_L^+ = x`, `a_R^- = 0`) the generalized equations imply:
- `(O_LR + O_RL.T) b_R^+ = 2 G_LL x`
- `2 G_LL a_L^- = (O_RL.T - O_LR) b_R^+`

Analogous equations hold for right incidence. Residuals below quantify algebraic consistency of the computed blocks.

In [ ]:
# Algebraic residual checks (left and right incidence).
N = len(modes_left)
x = np.zeros(N, dtype=np.complex128)
x[0] = 1.0
y = np.zeros(N, dtype=np.complex128)
y[0] = 1.0

b_from_left = T_LR @ x
a_refl_left = R_LL @ x
res_left_1 = np.linalg.norm((O_LR + O_RL.T) @ b_from_left - 2.0 * G_LL_reg @ x)
res_left_2 = np.linalg.norm(
    2.0 * G_LL_reg @ a_refl_left - (O_RL.T - O_LR) @ b_from_left
)

a_from_right = T_RL @ y
b_refl_right = R_RR @ y
res_right_1 = np.linalg.norm((O_RL + O_LR.T) @ a_from_right - 2.0 * G_RR_reg @ y)
res_right_2 = np.linalg.norm(
    2.0 * G_RR_reg @ b_refl_right - (O_LR.T - O_RL) @ a_from_right
)

print(f"left incidence residual 1:  {res_left_1:.3e}")
print(f"left incidence residual 2:  {res_left_2:.3e}")
print(f"right incidence residual 1: {res_right_1:.3e}")
print(f"right incidence residual 2: {res_right_2:.3e}")

In [ ]:
# Metric passivity diagnostic for the full interface map.
G_in = np.block(
    [[G_LL_reg, np.zeros((N, N), complex)], [np.zeros((N, N), complex), G_RR_reg]]
)
G_out = G_in.copy()
res_passive = metric_passivity_residual(S_if, G_in, G_out)
print(f"metric passivity residual lambda_max(S^H G S - G) = {res_passive:.3e}")

In [ ]:
# Optional comparison with 2-mode Lumerical reference (absolute values).
# Keep comparison like-for-like by truncating to N_ref modes PER port block.
S_ref_path = "assets/s_ref.npy"
try:
    S_ref = np.load(S_ref_path)
    N_ref = S_ref.shape[0] // 2
    N = len(modes_left)

    # For S_if = [[R_LL, T_RL], [T_LR, R_RR]], pick first N_ref modes on each port.
    idx = np.r_[np.arange(N_ref), N + np.arange(N_ref)]
    S_sub = S_if[np.ix_(idx, idx)]

    err_abs = np.linalg.norm(np.abs(S_sub) - np.abs(S_ref))
    print(f"S_ref shape: {S_ref.shape}, N_ref={N_ref}")
    print(f"S_sub shape: {S_sub.shape}")
    print(f"||abs(S_sub) - abs(S_ref)||_F = {err_abs:.3e}")
    print("abs(S_sub):")
    print(np.abs(S_sub))
    print("abs(S_ref):")
    print(np.abs(S_ref))
except FileNotFoundError:
    print(f"reference not found: {S_ref_path}")

In [ ]:
abs(S_sub - S_sub.T)

In [ ]:
abs(S_if[0, num_modes])

In [ ]:
abs(S_ref[0, 2])